# Runtime screening of the full method catalogue against reference implementations

Companion to `07_acsincome_runtime.ipynb`. Where notebook 07 studies three representatives across dataset sizes up to 1.5M samples, this notebook screens **every pre-processing method in the package** on ACSIncome subsets of 50,000 and 100,000 samples, and -- **whenever a public reference implementation exists** -- times it on the identical data:

| scikit-fair | Reference | Source |
|---|---|---|
| `Reweighing` | AIF360 | Kamiran & Calders released no code; AIF360 is the standard implementation |
| `FairBalance` | [hil-se/FairBalance](https://github.com/hil-se/FairBalance) | authors' code |
| `Massaging` | -- | no public original implementation |
| `DisparateImpactRemover` | `BlackBoxAuditing` | maintained by co-authors of Feldman et al.; the code AIF360 wraps |
| `FairwayRemover` | [joymallyac/Fairway](https://github.com/joymallyac/Fairway) | authors' notebook flow, reproduced verbatim |
| `FairMask` | -- | the released xFAIR code is an end-to-end evaluation loop (splits, internal SMOTE, metrics) with no separable pre-processing entry point |
| `FairOversampling` | [dd1github/Fair-Over-Sampling](https://github.com/dd1github/Fair-Over-Sampling) | authors' `FOS_1`+`FOS_2` flow from `FOS_main.py` |
| `HeterogeneousFOS` | -- | no public implementation |
| `FAWOS` | -- | authors' code is bound to their dataset abstraction; no standalone entry point |
| `FairSmote` | [joymallyac/Fair-SMOTE](https://github.com/joymallyac/Fair-SMOTE) | authors' code |
| `LearningFairRepresentations` | AIF360 | derived from Zemel et al.'s released code |

Protocol notes:

- Identical DataFrames on both sides; both implementations receive equivalent configurations (e.g., the same logistic-regression settings for Fairway; the AIF360 LFR hyperparameters match the package defaults, which follow the original publication).
- Only the pre-processing call is timed; input conversion (e.g., AIF360's `BinaryLabelDataset` construction) happens outside the timer.
- Each measurement is a single run: this is a screening pass, not a benchmark.
- `OptimizedPreproc` is excluded entirely (dataset-specific distortion constraints; cost governed by the discrete value domain, not by n), as are the `ReweighingClassifier`/`FairBalanceClassifier` wrappers (their pre-processing step is the `Reweighing`/`FairBalance` computation screened here).
- `FairMask`'s `fit` includes training its internal models, being a meta-estimator.

## 0. Setup

```bash
pip install scikit-fair BlackBoxAuditing aif360
```

The cell below clones the author repositories (pin commits to freeze the exact reference code).

In [1]:
import logging
import subprocess
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

# reference code triggers these per generated sample / per row; keep the log readable
warnings.filterwarnings("ignore", message="X does not have valid feature names")
logging.getLogger("smote_variants").setLevel(logging.WARNING)

from skfair.datasets import fetch_acs_income
from skfair.preprocessing import (
    FAWOS,
    DisparateImpactRemover,
    FairBalance,
    FairMask,
    FairOversampling,
    FairSmote,
    FairwayRemover,
    HeterogeneousFOS,
    LearningFairRepresentations,
    Massaging,
    Reweighing,
)

RANDOM_STATE = 42
SIZES = [50_000, 100_000]
LABEL = "Probability"  # label column name used by several reference code bases

REF_DIR = Path("reference_impls")
REPOS = {
    "Fair-SMOTE": ("https://github.com/joymallyac/Fair-SMOTE.git", None),
    "FairBalance": ("https://github.com/hil-se/FairBalance.git", None),
    "Fairway": ("https://github.com/joymallyac/Fairway.git", None),
    "Fair-Over-Sampling": ("https://github.com/dd1github/Fair-Over-Sampling.git", None),
}
for name, (url, commit) in REPOS.items():
    dest = REF_DIR / name
    if not dest.exists():
        subprocess.run(["git", "clone", url, str(dest)], check=True)
    if commit is not None:
        subprocess.run(["git", "-C", str(dest), "checkout", commit], check=True)


def subset(n):
    X, y = fetch_acs_income(subsample=n, random_state=RANDOM_STATE)
    return X, np.asarray(y)


DATA = {n: subset(n) for n in SIZES}
REPAIR_COLS = [c for c in DATA[SIZES[0]][0].columns if c not in ("SEX", "RAC1P")]
print({n: X.shape for n, (X, y) in DATA.items()})

{50000: (50000, 10), 100000: (100000, 10)}


## 1. Reference adapters

Each adapter reproduces the authors' own usage of their code (the Fairway loop and the FOS subgroup/argument computation are taken verbatim from their notebooks/`FOS_main.py`; the only substitution is `DataFrame.append` -> `pd.concat`, removed in pandas 2). `prepare` builds the inputs outside the timer; `run` is the timed call.

In [2]:
sys.path.insert(0, str(REF_DIR / "FairBalance" / "src"))
sys.path.insert(0, str(REF_DIR / "Fair-SMOTE"))
sys.path.insert(0, str(REF_DIR / "Fair-Over-Sampling"))

import Fair_OS as sv  # noqa: E402
from Generate_Samples import generate_samples  # noqa: E402
from preprocessor import FairBalance as fairbalance_reference  # noqa: E402
from aif360.algorithms.preprocessing import LFR  # noqa: E402
from aif360.algorithms.preprocessing import Reweighing as ReweighingAIF360  # noqa: E402
from aif360.datasets import BinaryLabelDataset  # noqa: E402
from BlackBoxAuditing.repairers.GeneralRepairer import Repairer  # noqa: E402

UNPRIV, PRIV = [{"SEX": 0}], [{"SEX": 1}]


def _bld(X, y):
    df = X.copy()
    df[LABEL] = y
    return BinaryLabelDataset(
        df=df, label_names=[LABEL], protected_attribute_names=["SEX"],
        favorable_label=1, unfavorable_label=0,
    )


def fairway_reference(df):
    # verbatim flow of Fairway/Split_On_Protected_Attribute/Adult.ipynb
    df = df.copy()
    male = df[df["SEX"] == 1].copy()
    female = df[df["SEX"] == 0].copy()
    male["SEX"] = 0
    female["SEX"] = 0
    lr = dict(C=1.0, penalty="l2", solver="liblinear", max_iter=100)
    clf_male = LogisticRegression(**lr).fit(
        male.loc[:, male.columns != LABEL], male[LABEL])
    clf_female = LogisticRegression(**lr).fit(
        female.loc[:, female.columns != LABEL], female[LABEL])
    df_removed = pd.DataFrame(columns=df.columns)
    for index, row in df.iterrows():
        row_ = [row.values[0 : len(row.values) - 1]]
        y_male = clf_male.predict(row_)
        y_female = clf_female.predict(row_)
        if y_male[0] != y_female[0]:
            df_removed = pd.concat(
                [df_removed, row.to_frame().T], ignore_index=True)
            df = df.drop(index)
    return df


def fairsmote_reference(df):
    # cell-balancing flow of the authors' notebooks (e.g. Adult_Sex.ipynb)
    cells = {
        (c, g): df[(df[LABEL] == c) & (df["SEX"] == g)]
        for c in (0, 1) for g in (0, 1)
    }
    target = max(len(cell) for cell in cells.values())
    grown = []
    for cell in cells.values():
        deficit = target - len(cell)
        if deficit > 0:
            cell_grown = generate_samples(deficit, cell.copy(), "")
            cell_grown.columns = cell.columns  # integer names for unknown df_name
            cell = cell_grown
        grown.append(cell)
    return pd.concat(grown, ignore_index=True)


def fos_reference(X, y):
    # subgroup counts + verbatim branching + FOS_1/FOS_2 flow of FOS_main.py
    Xa = X.to_numpy(dtype=float)
    ya = np.asarray(y, dtype=float)
    prot_idx = list(X.columns).index("SEX")
    pv = Xa[:, prot_idx]
    pv_max, pv_min = np.max(pv), np.min(pv)
    pv_mid_pt = pv_max - (pv_max + abs(pv_min)) / 2
    is_p = pv > pv_mid_pt
    n_fav, n_unfav = int((ya == 1).sum()), int((ya == 0).sum())
    n_p_fav = int((is_p & (ya == 1)).sum())
    n_p_unfav = int((is_p & (ya == 0)).sum())
    n_up_fav = int((~is_p & (ya == 1)).sum())
    n_up_unfav = int((~is_p & (ya == 0)).sum())
    majority = 1 if n_fav >= n_unfav else 0
    if n_p_fav < n_p_unfav:
        nsamp1, prot_grp1 = n_p_unfav - n_p_fav, 1
        cls_trk1 = 1 if majority == 1 else 0
    else:
        nsamp1, prot_grp1 = n_p_fav - n_p_unfav, 1
        cls_trk1 = 0 if majority == 1 else 1
    if n_up_fav < n_up_unfav:
        nsamp2, prot_grp2 = n_up_unfav - n_up_fav, 0
        cls_trk2 = 1 if majority == 1 else 0
    else:
        nsamp2, prot_grp2 = n_up_fav - n_up_unfav, 0
        cls_trk2 = 0 if majority == 1 else 1
    if nsamp1 < nsamp2:
        first, second = (nsamp1, cls_trk1, prot_grp1), (nsamp2, cls_trk2, prot_grp2)
    else:
        first, second = (nsamp2, cls_trk2, prot_grp2), (nsamp1, cls_trk1, prot_grp1)
    X1, y1 = sv.FOS_1().sample(Xa, ya, prot_idx, pv_mid_pt, first[2], first[1],
                               first[0], pv_max, pv_min)
    return sv.FOS_2().sample(X1, y1, prot_idx, pv_mid_pt, second[2], second[1], second[0])


def _df_with_label(X, y):
    df = X.copy()
    df[LABEL] = y
    return df


# name -> (prepare(X, y) -> args, run(*args))
REFERENCES = {
    "Reweighing": (
        lambda X, y: (_bld(X, y),),
        lambda bld: ReweighingAIF360(
            unprivileged_groups=UNPRIV, privileged_groups=PRIV).fit_transform(bld),
    ),
    "FairBalance": (
        lambda X, y: (X, y),
        lambda X, y: fairbalance_reference(X, y, ["SEX"]),
    ),
    "DisparateImpactRemover": (
        lambda X, y: (X[REPAIR_COLS + ["SEX"]].values.tolist(),),
        lambda data: Repairer(data, len(REPAIR_COLS), 1.0, False).repair(data),
    ),
    "FairwayRemover": (
        lambda X, y: (_df_with_label(X, y),),
        fairway_reference,
    ),
    "FairOversampling": (
        lambda X, y: (X, y),
        fos_reference,
    ),
    "FairSmote": (
        lambda X, y: (_df_with_label(X, y),),
        fairsmote_reference,
    ),
    "LearningFairRepresentations": (
        lambda X, y: (_bld(X, y),),
        lambda bld: LFR(unprivileged_groups=UNPRIV, privileged_groups=PRIV,
                        k=5, Ax=0.01, Ay=1.0, Az=50.0, seed=RANDOM_STATE,
                        verbose=0).fit(bld, maxiter=5000, maxfun=5000).transform(bld),
    ),
}

pip install 'aif360[AdversarialDebiasing]'


pip install 'aif360[AdversarialDebiasing]'


pip install 'aif360[Reductions]'


pip install 'aif360[Reductions]'


pip install 'aif360[inFairness]'


pip install 'aif360[Reductions]'


## 2. Screening loop

The `scikit-fair` side uses the package's registry defaults (`sens_attr`/`priv_group` supplied as in experiments); Fairway receives the same logistic-regression configuration as the authors' code.

In [3]:
FAIRWAY_LR = dict(C=1.0, penalty="l2", solver="liblinear", max_iter=100)

METHODS = {
    "Reweighing": (lambda: Reweighing(sens_attr="SEX"), "weight"),
    "FairBalance": (lambda: FairBalance(sens_attr="SEX"), "weight"),
    "Massaging": (lambda: Massaging(sens_attr="SEX", priv_group=1), "resample"),
    "DisparateImpactRemover": (
        lambda: DisparateImpactRemover(
            sens_attr="SEX", repair_columns=REPAIR_COLS, lambda_param=1.0
        ),
        "transform",
    ),
    "FairwayRemover": (
        lambda: FairwayRemover(sens_attr="SEX", priv_group=1,
                               estimator=LogisticRegression(**FAIRWAY_LR)),
        "resample",
    ),
    "FairMask": (lambda: FairMask(sens_attr="SEX", random_state=RANDOM_STATE), "fit"),
    "FairOversampling": (
        lambda: FairOversampling(
            sens_attr="SEX", priv_group=1, random_state=RANDOM_STATE
        ),
        "resample",
    ),
    "HeterogeneousFOS": (
        lambda: HeterogeneousFOS(sens_attr="SEX", random_state=RANDOM_STATE),
        "resample",
    ),
    "FAWOS": (
        lambda: FAWOS(sens_attr="SEX", priv_group=1, random_state=RANDOM_STATE),
        "resample",
    ),
    "FairSmote": (
        lambda: FairSmote(sens_attr="SEX", random_state=RANDOM_STATE),
        "resample",
    ),
    "LearningFairRepresentations": (
        lambda: LearningFairRepresentations(
            sens_attr="SEX", priv_group=1, random_state=RANDOM_STATE
        ),
        "transform",
    ),
}


def run_skfair(method_factory, kind, X, y):
    m = method_factory()
    t0 = time.perf_counter()
    if kind == "resample":
        m.fit_resample(X, y)
    elif kind == "transform":
        m.fit(X, y).transform(X)
    elif kind == "weight":
        m.fit_transform(X, y)
    elif kind == "fit":
        m.fit(X, y)
    return time.perf_counter() - t0


rows = []
for name, (factory, kind) in METHODS.items():
    for impl in ("scikit-fair", "reference"):
        if impl == "reference" and name not in REFERENCES:
            continue
        for n in SIZES:
            X, y = DATA[n]
            try:
                if impl == "scikit-fair":
                    seconds = run_skfair(factory, kind, X, y)
                else:
                    prepare, run = REFERENCES[name]
                    args = prepare(X, y)
                    t0 = time.perf_counter()
                    run(*args)
                    seconds = time.perf_counter() - t0
                rows.append({"method": name, "impl": impl, "n": n, "seconds": seconds})
                print(f"{name:28s} {impl:12s} n={n:>8,d}  {seconds:9.2f}s", flush=True)
            except Exception as exc:
                rows.append({"method": name, "impl": impl, "n": n, "seconds": np.nan})
                print(f"{name:28s} {impl:12s} n={n:>8,d}  FAILED: {exc!r}", flush=True)

Reweighing                   scikit-fair  n=  50,000       0.02s


Reweighing                   scikit-fair  n= 100,000       0.03s


Reweighing                   reference    n=  50,000       0.06s


Reweighing                   reference    n= 100,000       0.13s


FairBalance                  scikit-fair  n=  50,000       0.02s


...

FairSmote                    reference    n= 100,000      63.45s


LearningFairRepresentations  scikit-fair  n=  50,000     114.76s


LearningFairRepresentations  scikit-fair  n= 100,000     317.19s


LearningFairRepresentations  reference    n=  50,000     108.38s


LearningFairRepresentations  reference    n= 100,000     394.38s


In [4]:
res = pd.DataFrame(rows)
wide = res.pivot_table(index="method", columns=["impl", "n"], values="seconds", sort=False)
wide = wide.reindex(list(METHODS))
if ("reference", SIZES[-1]) in wide.columns:
    wide[("ratio", SIZES[-1])] = (
        wide[("scikit-fair", SIZES[-1])] / wide[("reference", SIZES[-1])]
    )

out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)
res.to_csv(out_dir / "acsincome_screening.csv", index=False)

wide.round(2)

impl                        scikit-fair         reference          ratio
n                                50000   100000    50000   100000 100000
method                                                                  
Reweighing                         0.02    0.03      0.06    0.13   0.22
FairBalance                        0.02    0.03      0.56    1.09   0.02
Massaging                          0.17    0.40       NaN     NaN    NaN
DisparateImpactRemover             0.23    0.41      1.82    3.91   0.11
FairwayRemover                     0.13    0.27     76.64  199.64   0.00
FairMask                          27.35   78.37       NaN     NaN    NaN
FairOversampling                   1.16    3.30      2.56    7.15   0.46
HeterogeneousFOS                  51.79  111.23       NaN     NaN    NaN
FAWOS                             13.23   31.42       NaN     NaN    NaN
FairSmote                         31.68   64.50     31.65   63.45   1.02
LearningFairRepresentations      114.76  317.19    108.38  394.38   0.80

## Reading the table

`ratio` is `scikit-fair` / reference at 100k samples (values at or below 1 mean the unified implementation is at least as fast as the code released for the method). Methods without a `reference` column have no publicly runnable original implementation, as detailed in the introduction; for those, the two sizes still verify that the implementation scales linearly.

Preparing this screening led to internal performance work in the package -- vectorised weight assignment in `Reweighing` and batched synthetic-sample generation in `FairOversampling` -- in both cases preserving the method's procedure (see `CHANGELOG.md`).